# Classifying `[CLS]` embeddings: train → test → classify

A logistic regression learns to tell four kinds of medical titles apart
(**neoplasms, digestive, nervous, cardiovascular**) from their `[CLS]`
vectors, and then classifies the MultiCaRe texts of
`../embedding/text-embeddings.ipynb`.

The same steps with more analysis in between (padding and batches,
per-class report, confusion matrix, errors, how many examples are needed):
`title-classification-in-depth.ipynb`.

## 1. Preparation

The model of the embedding notebook, the texts to classify, and the
labeled titles (PubMed titles from the *Medical Abstracts* corpus,
downloaded and cleaned by `load_titles()`). We take 600 titles per class
and split them: two thirds to train, one third to test.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
import transformers
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from classification_helpers import load_titles

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()

MODEL_NAME = "BAAI/bge-small-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval()

TEXTS = {
    "Text 1": "A 44-year-old woman presented with a 3-day history of right flank and lower quadrant "
              "abdominal pain associated with nausea and constipation.",
    "Text 2": "A 41-year-old female with a history of cesarean section presented to the emergency "
              "department with continuous abdominal pain for 12 h, localized in the right iliac fossa (RIF).",
    "Text 3": "A 23-year-old male was brought to the emergency of Hamad General Hospital in Doha, after "
              "being found on the floor dizzy and confused, with urinary incontinence although no one "
              "witnessed convulsions.",
    "Text 4": "However, 10 days later, she developed a non-pruritic morbilliform rash spreading from trunk "
              "to the entire body, including palms, soles, and face with mild lip and eye swelling.",
}

titles = load_titles()
sample = titles.groupby("label").sample(600, random_state=0)
train, test = train_test_split(sample, test_size=1/3, stratify=sample.label, random_state=0)
print("train:", len(train), "titles   test:", len(test), "titles")

14,438 rows → 11,227 distinct abstracts → 8,298 with a single label → 5,683 titles in 4 classes
train: 1600 titles   test: 800 titles


## 2. Titles → embeddings

The basic cycle of the embedding notebook (tokenize, encode, keep
`[CLS]`), applied to every title.

In [2]:
def embed(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[0, 0].numpy()

X_train = [embed(t) for t in train.title]
X_test = [embed(t) for t in test.title]

## 3. Training

The classifier sees the training embeddings with their labels.

In [3]:
classifier = LogisticRegression(max_iter=1000).fit(X_train, train.label)

## 4. Testing

**Accuracy** on the test titles, which the classifier never saw: the
fraction it labels correctly (by chance it would be 0.25).

In [4]:
print("accuracy:", round(classifier.score(X_test, test.label), 3))

accuracy: 0.865


## 5. Classifying new texts

In [5]:
for name, text in TEXTS.items():
    probabilities = classifier.predict_proba([embed(text)])[0]
    best = probabilities.argmax()
    print(f"{name} → {classifier.classes_[best]}  (P = {probabilities[best]:.3f})")

Text 1 → digestive  (P = 0.999)
Text 2 → digestive  (P = 0.997)
Text 3 → nervous  (P = 0.705)
Text 4 → digestive  (P = 0.973)


## What to notice

* **Only the classifier learns.** The model that makes the embeddings is
  used as it is; training fits the logistic regression's weights.
* **0.865 on unseen titles**, against 0.25 by chance.
* **Texts 1 and 2 → digestive, text 3 → nervous**, although the classifier
  learned from PubMed titles and never saw a case report. Text 3 is the
  least sure (P = 0.705).
* **Text 4 → digestive with P = 0.973**, but a rash fits none of the four
  classes. The classifier always picks one of the classes it knows: a high
  probability means "more like this class than the other three", not
  "surely this class".

(`title-classification-in-depth.ipynb` gives 0.864: it embeds the titles in
batches, which changes the vectors only by rounding.)

## Credits

* Data: T. Schopf, D. Braun, F. Matthes. *Evaluating Unsupervised Text
  Classification: Zero-shot and Similarity-based Approaches.* NLPIR 2022.
  [Medical Abstracts TC Corpus](https://github.com/sebischair/Medical-Abstracts-TC-Corpus),
  CC BY-SA 3.0.
* Model: [`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-small-en-v1.5).
* The four texts: MultiCaRe cases `PMC5137649_01`, `PMC11722600_01`,
  `PMC6949661_01`, `PMC6134951_01` (sample in `../../data/multicare/data/csv_sample/`).